<a href="https://colab.research.google.com/github/rania-elfadili/python-IA/blob/main/TP2_TF_IDF_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# importation des bibliotheques importants
import nltk
import math
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Téléchargement des ressources nécessaires
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


# On définit un petit corpus composé de 3 phrases simples pour tester
corpus = [
    "Natural Language Processing is interesting",
    "Machine learning helps in NLP tasks",
    "Text representation is important in NLP"
]


# Initialisation des outils NLP
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

processed_sentences = []

# On applique les étapes de prétraitement sur chaque phrase
for sentence in corpus:
    tokens = word_tokenize(sentence)              # Tokenisation
    tokens = [w.lower() for w in tokens]          # Mise en minuscule
    tokens = [w for w in tokens if w.isalpha()    # Suppression ponctuation
              and w not in stop_words]            # Suppression stop words
    tokens = [lemmatizer.lemmatize(w) for w in tokens]  # Lemmatisation
    processed_sentences.append(tokens)

print("\n=== PHRASES APRES PRETRAITEMENT ===")
for s in processed_sentences:
    print(s)


# On crée un vocabulaire contenant tous les mots uniques
vocab = sorted(list(set(word for sent in processed_sentences for word in sent)))

# On associe chaque mot à un index numérique
word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}

print("\n=== VOCABULAIRE ===")
print(word_to_index)

# PARTIE 1
# Chaque mot est représenté par un vecteur avec un seul 1
print("\n=== ONE-HOT ENCODING ===")

def one_hot(word):
    vec = [0] * len(vocab)
    vec[word_to_index[word]] = 1
    return vec

# Affichage des vecteurs one-hot
for word in vocab:
    print(word, one_hot(word))

#  PARTIE 2
# Chaque phrase est représentée par un vecteur de fréquences
print("\n=== BAG OF WORDS ===")

def bag_of_words(sentence):
    vec = [0] * len(vocab)
    for word in sentence:
        vec[word_to_index[word]] += 1  # Comptage des occurrences
    return vec

# Affichage des vecteurs BoW pour chaque phrase
for i, sentence in enumerate(processed_sentences):
    print(f"Phrase {i+1}:", bag_of_words(sentence))

#  PARTIE 3
print("\n=== TF-IDF ===")

#  Calcul TF (fréquence des mots dans chaque phrase)
tf_matrix = []
for sentence in processed_sentences:
    vec = [0] * len(vocab)
    for word in sentence:
        vec[word_to_index[word]] += 1
    tf_matrix.append(vec)

print("\nTF :")
for row in tf_matrix:
    print(row)

#  Calcul IDF (importance du mot dans le corpus)
N = len(processed_sentences)  # Nombre de phrases
idf_vector = []

for word in vocab:
    df = sum(1 for sentence in processed_sentences if word in sentence)
    idf = math.log(N / df)  # Formule IDF
    idf_vector.append(idf)

print("\nIDF :")
print(idf_vector)

#  Calcul TF-IDF
tfidf_matrix = []
for row in tf_matrix:
    tfidf_row = [row[i] * idf_vector[i] for i in range(len(vocab))]
    tfidf_matrix.append(tfidf_row)

print("\nTF-IDF :")
for row in tfidf_matrix:
    print(row)

#  PARTIE 4
# On implémente deux modèles : CBOW et Skip-gram

sentences = processed_sentences
embedding_dim = 5   # Taille des vecteurs
window_size = 2     # Taille du contexte

# Génération données CBOW
def generate_cbow_data(sentences):
    data = []
    for sentence in sentences:
        for i in range(window_size, len(sentence) - window_size):
            context = []
            for j in range(i - window_size, i + window_size + 1):
                if j != i:
                    context.append(word_to_index[sentence[j]])
            target = word_to_index[sentence[i]]
            data.append((context, target))
    return data

#  Génération données Skip-gram
def generate_skipgram_data(sentences):
    data = []
    for sentence in sentences:
        for i, word in enumerate(sentence):
            for j in range(i - window_size, i + window_size + 1):
                if j != i and 0 <= j < len(sentence):
                    data.append((word_to_index[word], word_to_index[sentence[j]]))
    return data

cbow_data = generate_cbow_data(sentences)
skipgram_data = generate_skipgram_data(sentences)

#  MODELE CBOW
class CBOW(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(vocab), embedding_dim)
        self.linear = nn.Linear(embedding_dim, len(vocab))

    def forward(self, x):
        x = self.emb(x)
        x = torch.mean(x, dim=0).view(1, -1)
        return self.linear(x)

cbow_model = CBOW()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(cbow_model.parameters(), lr=0.01)

# Entraînement du modèle CBOW
for epoch in range(100):
    for context, target in cbow_data:
        context = torch.tensor(context)
        target = torch.tensor([target])

        optimizer.zero_grad()
        loss = loss_fn(cbow_model(context), target)
        loss.backward()
        optimizer.step()

print("\n=== EMBEDDINGS CBOW ===")
for word in vocab:
    print(word, cbow_model.emb.weight[word_to_index[word]].detach().numpy())

#  MODELE SKIP-GRAM
class SkipGram(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(vocab), embedding_dim)
        self.linear = nn.Linear(embedding_dim, len(vocab))

    def forward(self, x):
        x = self.emb(x).view(1, -1)
        return self.linear(x)

sg_model = SkipGram()
optimizer = optim.Adam(sg_model.parameters(), lr=0.01)

# Entraînement Skip-gram
for epoch in range(100):
    for target, context in skipgram_data:
        target = torch.tensor([target])
        context = torch.tensor([context])

        optimizer.zero_grad()
        loss = loss_fn(sg_model(target), context)
        loss.backward()
        optimizer.step()

print("\n=== EMBEDDINGS SKIP-GRAM ===")
for word in vocab:
    print(word, sg_model.emb.weight[word_to_index[word]].detach().numpy())


# Permet de mesurer la proximité entre deux mots

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\n=== SIMILARITE ENTRE MOTS ===")

pairs = [("nlp", "learning"), ("language", "processing")]

for w1, w2 in pairs:
    v1 = cbow_model.emb.weight[word_to_index[w1]].detach().numpy()
    v2 = cbow_model.emb.weight[word_to_index[w2]].detach().numpy()
    print(f"{w1} - {w2} :", cosine_similarity(v1, v2))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rania\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rania\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rania\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



=== PHRASES APRES PRETRAITEMENT ===
['natural', 'language', 'processing', 'interesting']
['machine', 'learning', 'help', 'nlp', 'task']
['text', 'representation', 'important', 'nlp']

=== VOCABULAIRE ===
{'help': 0, 'important': 1, 'interesting': 2, 'language': 3, 'learning': 4, 'machine': 5, 'natural': 6, 'nlp': 7, 'processing': 8, 'representation': 9, 'task': 10, 'text': 11}

=== ONE-HOT ENCODING ===
help [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
important [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
interesting [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
language [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
learning [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
machine [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
natural [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]
nlp [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
processing [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
representation [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
task [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
text [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]

=== BAG OF WORDS ===
Phrase 1: [0, 0, 1, 1, 0, 